# 40 CNN GNN LSTM

Uses clearance strips plus a small ego-uav graph sequence. Saves all artifacts to disk so each run survives a kernel restart.


In [1]:
import gc
import json
import sys
from pathlib import Path

import torch

gc.collect()
try:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
except Exception:
    pass

PROJECT_ROOT = Path.home() / "Documents/Thesis"
PIPELINE_ROOT = PROJECT_ROOT / "08_model_training_pipeline"
if str(PIPELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(PIPELINE_ROOT))

from training.notebook_workflow import (
    CNNGNNLSTMTrajectoryPredictor,
    CNNGNNLSTMTransformerTrajectoryPredictor,
    CNNGNNTransformerTrajectoryPredictor,
    CNNLSTMTrajectoryPredictor,
    GoalOnlyTrajectoryDataset,
    LSTMGoalTrajectoryPredictor,
    ScanGoalTrajectoryDataset,
    ScanGraphTrajectoryDataset,
    collate_goal_only,
    collate_scan,
    collate_scan_graph,
    device_from_flag,
    evaluate_trajectory_model,
    load_or_build_shared_artifacts,
    make_dataloaders,
    prepare_result_dirs,
    run_constant_velocity_baseline,
    save_final_trajectory_evaluation,
    set_seed,
    timestamp_tag,
    train_trajectory_model,
)


In [2]:
SEED = 42
PAST_LEN = 10
FUTURE_LEN = 5
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
MAX_SAMPLES = None
USE_CPU = False

BATCH_SIZE = 64
EPOCHS = 30
EARLY_STOPPING_PATIENCE = 5
LR = 1e-3
WEIGHT_DECAY = 1e-4

GOAL_DIM = 13
NODE_DIM = 12
EDGE_DIM = 8
HIDDEN_DIM = 128
GRAPH_HIDDEN = 96
DROPOUT = 0.10
MSG_PASSES = 2
TRANSFORMER_HEADS = 4
TRANSFORMER_LAYERS = 2
TRANSFORMER_FF = 256

device = device_from_flag(USE_CPU)
print("Device:", device)


Device: cuda


In [3]:
set_seed(SEED)
streams, sample_table, split_info, sample_table_path, split_path = load_or_build_shared_artifacts(
    past_len=PAST_LEN,
    future_len=FUTURE_LEN,
    seed=SEED,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
)
print("Sample table:", sample_table_path)
print("Split path:", split_path)
print("Split strategy:", split_info["split_strategy"])
print("Episode count:", split_info["episode_count"])
print("Train / Val / Test samples:", len(split_info["train_indices"]), len(split_info["val_indices"]), len(split_info["test_indices"]))
print("Train episodes:", split_info["train_episode_ids"])
print("Val episodes:", split_info["val_episode_ids"])
print("Test episodes:", split_info["test_episode_ids"])


Sample table: /home/basudeo/Documents/Thesis/08_model_training_pipeline/results/train_ready/sample_table_seed42_past10_future5.json
Split path: /home/basudeo/Documents/Thesis/08_model_training_pipeline/results/splits/trajectory_split_seed42_past10_future5.json
Split strategy: episode
Episode count: 6
Train / Val / Test samples: 29379 6195 8967
Train episodes: ['run_model_20260511_220103', 'run_model_20260511_210809', 'run_model_20260511_213251', 'run_model_20260511_221726']
Val episodes: ['run_model_20260511_202309']
Test episodes: ['run_model_20260511_222947']


In [4]:
MODEL_SLUG = "cnn_gnn_lstm"
TIMESTAMP = timestamp_tag()
result_dir, weight_dir, plot_dir = prepare_result_dirs(MODEL_SLUG)

dataset = ScanGraphTrajectoryDataset(streams, sample_table, PAST_LEN)
train_loader, val_loader, test_loader = make_dataloaders(
    dataset,
    split_info,
    batch_size=BATCH_SIZE,
    collate_fn=collate_scan_graph,
    max_samples=MAX_SAMPLES,
)

model = CNNGNNLSTMTrajectoryPredictor(
    goal_dim=GOAL_DIM,
    node_dim=NODE_DIM,
    edge_dim=EDGE_DIM,
    hidden_dim=HIDDEN_DIM,
    graph_hidden=GRAPH_HIDDEN,
    future_len=FUTURE_LEN,
    dropout=DROPOUT,
    msg_passes=MSG_PASSES,
).to(device)

train_out = train_trajectory_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    model_slug=MODEL_SLUG,
    timestamp=TIMESTAMP,
    split_path=split_path,
    sample_table_path=sample_table_path,
    result_dir=result_dir,
    weight_dir=weight_dir,
    plot_dir=plot_dir,
    epochs=EPOCHS,
    patience=EARLY_STOPPING_PATIENCE,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    extra_manifest={"family": "scan_graph_lstm", "future_len": FUTURE_LEN, "msg_passes": MSG_PASSES},
)
test_eval = evaluate_trajectory_model(model, test_loader, device)
final_metrics = save_final_trajectory_evaluation(
    model_slug=MODEL_SLUG,
    timestamp=TIMESTAMP,
    train_out=train_out,
    test_eval=test_eval,
    split_path=split_path,
    sample_table_path=sample_table_path,
    result_dir=result_dir,
    plot_dir=plot_dir,
)
print(json.dumps(final_metrics, indent=2))


[cnn_gnn_lstm] start: epochs=30 patience=5 train_batches=460 val_batches=97
[cnn_gnn_lstm] epoch 01/30 train_loss=0.000602 val_loss=0.000272 val_ADE=0.015010 val_FDE=0.020309 val_RMSE=0.033028 best_ADE=0.015010 status=improved
[cnn_gnn_lstm] epoch 02/30 train_loss=0.000468 val_loss=0.000252 val_ADE=0.011960 val_FDE=0.011912 val_RMSE=0.031766 best_ADE=0.011960 status=improved
[cnn_gnn_lstm] epoch 03/30 train_loss=0.000461 val_loss=0.000248 val_ADE=0.011276 val_FDE=0.011255 val_RMSE=0.031512 best_ADE=0.011276 status=improved
[cnn_gnn_lstm] epoch 04/30 train_loss=0.000459 val_loss=0.000254 val_ADE=0.012375 val_FDE=0.011976 val_RMSE=0.031932 best_ADE=0.011276 status=no_improve(1/5)
[cnn_gnn_lstm] epoch 05/30 train_loss=0.000460 val_loss=0.000250 val_ADE=0.010958 val_FDE=0.009466 val_RMSE=0.031660 best_ADE=0.010958 status=improved
[cnn_gnn_lstm] epoch 06/30 train_loss=0.000459 val_loss=0.000254 val_ADE=0.011911 val_FDE=0.011079 val_RMSE=0.031894 best_ADE=0.010958 status=no_improve(1/5)
[cnn